In [0]:
from pyspark.sql import functions as F

patients = spark.table(
    "healthcare.default.silver_patients"
)

appointments = spark.table(
    "healthcare.default.silver_appointments"
)

treatments = spark.table(
    "healthcare.default.silver_treatments"
)

billing = spark.table(
    "healthcare.default.silver_billing"
)

print("Patients:", patients.count())
print("Appointments:", appointments.count())
print("Treatments:", treatments.count())
print("Billing:", billing.count())

Patients: 50
Appointments: 200
Treatments: 200
Billing: 200


In [0]:
patient_appointments = (
    appointments
    .groupBy("patient_id")
    .agg(
        F.count("appointment_id").alias("total_appointments"),
        F.sum(
            F.when(F.col("status") == "Completed", 1).otherwise(0)
        ).alias("completed_appointments"),
        F.sum(
            F.when(F.col("status") == "Cancelled", 1).otherwise(0)
        ).alias("cancelled_appointments"),
        F.sum(
            F.when(F.col("status") == "No-show", 1).otherwise(0)
        ).alias("no_show_appointments"),
        F.min("appointment_date").alias("first_appointment_date"),
        F.max("appointment_date").alias("last_appointment_date")
    )
)

display(patient_appointments.orderBy("patient_id"))

patient_id,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date
P001,4,0,1,1,2023-01-16,2023-04-09
P002,3,0,1,0,2023-01-20,2023-10-06
P003,2,0,0,0,2023-08-16,2023-08-26
P004,2,2,0,0,2023-02-04,2023-07-07
P005,8,2,0,4,2023-01-01,2023-11-14
P007,4,0,3,0,2023-01-07,2023-12-30
P008,2,1,0,0,2023-04-06,2023-05-02
P009,4,1,1,1,2023-03-21,2023-10-22
P010,6,1,1,3,2023-03-27,2023-09-28
P011,2,1,1,0,2023-04-18,2023-07-04


In [0]:
patient_treatments = (
    treatments
    .groupBy("appointment_id")
    .agg(
        F.count("treatment_id").alias("total_treatments"),
        F.sum("cost").alias("total_treatment_cost"),
        F.avg("cost").alias("average_treatment_cost"),
        F.min("treatment_date").alias("first_treatment_date"),
        F.max("treatment_date").alias("last_treatment_date")
    )
)

display(
    patient_treatments
    .orderBy("appointment_id")
)

appointment_id,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
A001,1,3941.97,3941.97,2023-08-09,2023-08-09
A002,1,4158.44,4158.44,2023-06-09,2023-06-09
A003,1,3731.55,3731.55,2023-06-28,2023-06-28
A004,1,4799.86,4799.86,2023-09-01,2023-09-01
A005,1,582.05,582.05,2023-07-06,2023-07-06
A006,1,1381.0,1381.0,2023-06-19,2023-06-19
A007,1,534.03,534.03,2023-04-09,2023-04-09
A008,1,3413.64,3413.64,2023-05-24,2023-05-24
A009,1,4541.14,4541.14,2023-03-05,2023-03-05
A010,1,1595.67,1595.67,2023-01-13,2023-01-13


In [0]:
patient_billing = (
    billing
    .groupBy("patient_id")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_billed_amount"),
        F.avg("amount").alias("average_bill_amount"),
        F.sum(
            F.when(F.col("payment_status") == "Paid", 1).otherwise(0)
        ).alias("paid_bills"),
        F.sum(
            F.when(F.col("payment_status") == "Pending", 1).otherwise(0)
        ).alias("pending_bills"),
        F.sum(
            F.when(F.col("payment_status") == "Failed", 1).otherwise(0)
        ).alias("failed_bills"),
        F.min("bill_date").alias("first_bill_date"),
        F.max("bill_date").alias("last_bill_date")
    )
)

display(
    patient_billing
    .orderBy("patient_id")
)

patient_id,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date
P001,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09
P002,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06
P003,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26
P004,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07
P005,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14
P007,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30
P008,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02
P009,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22
P010,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28
P011,2,7728.880000000001,3864.4400000000005,0,1,1,2023-04-18,2023-07-04


In [0]:
# Combine patient, appointment, treatment, and billing summaries

gold_patient_summary = (
    patients
    .select(
        "patient_id",
        "first_name",
        "last_name",
        "gender",
        "date_of_birth",
        "registration_date",
        "insurance_provider"
    )

    # Add appointment metrics
    .join(
        patient_appointments,
        on="patient_id",
        how="left"
    )

    # Add billing metrics
    .join(
        patient_billing,
        on="patient_id",
        how="left"
    )
)

# Replace NULL metrics with zero where a patient has no activity
gold_patient_summary = (
    gold_patient_summary
    .fillna({
        "total_appointments": 0,
        "completed_appointments": 0,
        "cancelled_appointments": 0,
        "no_show_appointments": 0,
        "total_bills": 0,
        "total_billed_amount": 0.0,
        "average_bill_amount": 0.0,
        "paid_bills": 0,
        "pending_bills": 0,
        "failed_bills": 0
    })
)

print(
    "Gold patient summary rows:",
    gold_patient_summary.count()
)

display(
    gold_patient_summary
    .orderBy("patient_id")
)

Gold patient summary rows: 50


patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date
P001,David,Williams,Female,1955-06-04,2022-06-23,WellnessCorp,4,0,1,1,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09
P002,Emily,Smith,Female,1984-10-12,2022-01-15,PulseSecure,3,0,1,0,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06
P003,Laura,Jones,Male,1977-08-21,2022-02-07,PulseSecure,2,0,0,0,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26
P004,Michael,Johnson,Female,1981-02-20,2021-03-02,HealthIndia,2,2,0,0,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07
P005,David,Wilson,Male,1960-06-23,2021-09-29,MedCare Plus,8,2,0,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14
P006,Linda,Jones,Male,1963-06-16,2022-10-02,HealthIndia,0,0,0,0,null,null,0,0.0,0.0,0,0,0,null,null
P007,Alex,Johnson,Female,1989-06-08,2021-12-25,MedCare Plus,4,0,3,0,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30
P008,David,Davis,Female,1976-07-05,2021-05-25,WellnessCorp,2,1,0,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02
P009,Laura,Davis,Male,1971-12-11,2022-09-18,PulseSecure,4,1,1,1,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22
P010,Michael,Taylor,Male,2001-10-13,2022-08-24,WellnessCorp,6,1,1,3,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28


In [0]:
patient_treatment_summary = (
    treatments
    .join(
        appointments.select(
            "appointment_id",
            "patient_id"
        ).distinct(),
        on="appointment_id",
        how="inner"
    )
    .groupBy("patient_id")
    .agg(
        F.count("treatment_id").alias("total_treatments"),
        F.sum("cost").alias("total_treatment_cost"),
        F.avg("cost").alias("average_treatment_cost"),
        F.min("treatment_date").alias("first_treatment_date"),
        F.max("treatment_date").alias("last_treatment_date")
    )
)

display(
    patient_treatment_summary
    .orderBy("patient_id")
)

patient_id,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,8,18609.91,2326.23875,2023-01-01,2023-11-14
P007,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28
P011,2,7728.880000000001,3864.4400000000005,2023-04-18,2023-07-04


In [0]:
gold_patient_summary = (
    gold_patient_summary
    .join(
        patient_treatment_summary,
        on="patient_id",
        how="left"
    )
)

gold_patient_summary = (
    gold_patient_summary
    .fillna({
        "total_treatments": 0,
        "total_treatment_cost": 0.0,
        "average_treatment_cost": 0.0
    })
)

print(
    "Final Gold patient summary rows:",
    gold_patient_summary.count()
)

display(
    gold_patient_summary
    .orderBy("patient_id")
)


Final Gold patient summary rows: 50


patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,David,Williams,Female,1955-06-04,2022-06-23,WellnessCorp,4,0,1,1,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,Emily,Smith,Female,1984-10-12,2022-01-15,PulseSecure,3,0,1,0,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,Laura,Jones,Male,1977-08-21,2022-02-07,PulseSecure,2,0,0,0,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,Michael,Johnson,Female,1981-02-20,2021-03-02,HealthIndia,2,2,0,0,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,David,Wilson,Male,1960-06-23,2021-09-29,MedCare Plus,8,2,0,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2023-01-01,2023-11-14
P006,Linda,Jones,Male,1963-06-16,2022-10-02,HealthIndia,0,0,0,0,null,null,0,0.0,0.0,0,0,0,null,null,0,0.0,0.0,null,null
P007,Alex,Johnson,Female,1989-06-08,2021-12-25,MedCare Plus,4,0,3,0,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,David,Davis,Female,1976-07-05,2021-05-25,WellnessCorp,2,1,0,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,Laura,Davis,Male,1971-12-11,2022-09-18,PulseSecure,4,1,1,1,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,Michael,Taylor,Male,2001-10-13,2022-08-24,WellnessCorp,6,1,1,3,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28


In [0]:
gold_patient_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.gold_patient_summary"
    )

print("gold_patient_summary created successfully")

gold_patient_summary created successfully


In [0]:
gold_patient_check = spark.table(
    "healthcare.default.gold_patient_summary"
)

print(
    "Gold patient summary count:",
    gold_patient_check.count()
)

display(
    gold_patient_check
    .orderBy("patient_id")
)

Gold patient summary count: 50


patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,David,Williams,Female,1955-06-04,2022-06-23,WellnessCorp,4,0,1,1,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,Emily,Smith,Female,1984-10-12,2022-01-15,PulseSecure,3,0,1,0,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,Laura,Jones,Male,1977-08-21,2022-02-07,PulseSecure,2,0,0,0,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,Michael,Johnson,Female,1981-02-20,2021-03-02,HealthIndia,2,2,0,0,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,David,Wilson,Male,1960-06-23,2021-09-29,MedCare Plus,8,2,0,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2023-01-01,2023-11-14
P006,Linda,Jones,Male,1963-06-16,2022-10-02,HealthIndia,0,0,0,0,null,null,0,0.0,0.0,0,0,0,null,null,0,0.0,0.0,null,null
P007,Alex,Johnson,Female,1989-06-08,2021-12-25,MedCare Plus,4,0,3,0,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,David,Davis,Female,1976-07-05,2021-05-25,WellnessCorp,2,1,0,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,Laura,Davis,Male,1971-12-11,2022-09-18,PulseSecure,4,1,1,1,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,Michael,Taylor,Male,2001-10-13,2022-08-24,WellnessCorp,6,1,1,3,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28


In [0]:
from pyspark.sql import functions as F

doctors = spark.table(
    "healthcare.default.silver_doctors"
)

display(doctors)

doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,valid_doctor_id,valid_names,valid_specialization,valid_experience,valid_email,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash
D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,2fd108434c9a1abca8ee2737545b65c5821dbbecf358ad2f1512e6c897fef35d
D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a13a9beb52ea33d58dd271aab12aa727346b2991cc8d2bc6e914edb26526d92
D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a57399a935744dcdb43182b557c416c61f69873ca546b319b2ae58a376f4756f
D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,537e428029b87c3e1ddc93b1da665d8859b77aed5a6836c238abab4954054211
D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,382a00030c2f49febb0f65dba0c7b8d75732e4ffcd194dd4eef46158578f1918
D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,5a4ee9fb29deb16f3c2cb56b241c10ba8a4b1582209f16a58b919e04121fcc61
D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,a3f6bc87dd0e29cc01bdd5978b2809a7c34ecd3770df10012475d7e50771b2cf
D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,8f33e2856b4bd096bd8fa18289d7717b97af798bd97473e31fe4e38480e1e755
D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,22eeaeffa938c89c7e5c7387605f929a52088af39d49979a9da048556717268c
D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com,true,true,true,true,true,BATCH_20260810_144925_920fea,SRC004,doctors,doctors.csv,2026-08-10T14:49:39.143Z,2026-08-10,dcd6af3ca88c8a8fe020c530616c3c27724f056593c5b3f06056eb03d9994a91


In [0]:
# Load Silver tables
appointments = spark.table(
    "healthcare.default.silver_appointments"
)

patients = spark.table(
    "healthcare.default.silver_patients"
)

doctors = spark.table(
    "healthcare.default.silver_doctors"
)

treatments = spark.table(
    "healthcare.default.silver_treatments"
)

billing = spark.table(
    "healthcare.default.silver_billing"
)


# Create Gold Appointment Summary
gold_appointment_summary = (
    appointments
    .join(
        patients.select(
            "patient_id",
            "first_name",
            "last_name"
        ),
        on="patient_id",
        how="left"
    )
    .join(
        doctors.select(
            "doctor_id",
            F.concat_ws(
                " ",
                F.col("first_name"),
                F.col("last_name")
            ).alias("doctor_name"),
            "specialization"
        ),
        on="doctor_id",
        how="left"
    )
    .join(
        treatments.select(
            "treatment_id",
            "appointment_id",
            "treatment_type",
            "cost"
        ),
        on="appointment_id",
        how="left"
    )
    .join(
        billing.select(
            "treatment_id",
            "amount",
            "payment_method",
            "payment_status"
        ),
        on="treatment_id",
        how="left"
    )
)


print(
    "Gold appointment summary rows:",
    gold_appointment_summary.count()
)


display(
    gold_appointment_summary
    .orderBy("appointment_id")
)

Gold appointment summary rows: 200


treatment_id,appointment_id,doctor_id,patient_id,appointment_date,appointment_time,reason_for_visit,status,valid_appointment_id,valid_patient_id,valid_doctor_id,valid_appointment_date,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,first_name,last_name,doctor_name,specialization,treatment_type,cost,amount,payment_method,payment_status
T001,A001,D009,P034,2023-08-09,15:15:00,Therapy,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,39129462e5fd1bff89a54b86c6bd10cc636bf78ed464a8c833809f18a05ac71c,Alex,Smith,Sarah Smith,Pediatrics,Chemotherapy,3941.97,3941.97,Insurance,Pending
T002,A002,D004,P032,2023-06-09,14:30:00,Therapy,No-show,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,94e5618f556eda537cbd89100999833711c6ea8d3064b7cf71728df6c830bb0a,Alex,Moore,David Jones,Pediatrics,Mri,4158.44,4158.44,Insurance,Paid
T003,A003,D004,P048,2023-06-28,08:00:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f15b73ac6f2f662ecaa2e0cf1ade2a405df630fbec5a0845f0e67a5d387a989a,Emily,Miller,David Jones,Pediatrics,Mri,3731.55,3731.55,Insurance,Paid
T004,A004,D006,P025,2023-09-01,09:15:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,40c3a0117e75056382aff06888c18b72e193d7c745f3f3a2cf95335b170631bc,Robert,Wilson,Alex Davis,Pediatrics,Mri,4799.86,4799.86,Insurance,Failed
T005,A005,D003,P040,2023-07-06,12:45:00,Emergency,No-show,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9ee8014971d675609a0d868f8bcc68972d46e04991d189beb4daaa9ff3a7500c,Emily,Williams,Jane Smith,Pediatrics,Ecg,582.05,582.05,Credit Card,Pending
T006,A006,D006,P045,2023-06-19,16:15:00,Checkup,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,9df730a3521ae05c83de230fa69c9d1575dd4e4361d4c0aa22ed2d9fb1c60866,Linda,Miller,Alex Davis,Pediatrics,Chemotherapy,1381.0,1381.0,Insurance,Pending
T007,A007,D007,P001,2023-04-09,10:30:00,Consultation,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,3eb0a8c591025b2af9fe59b3a022f13bb637d30d693deead97866fde92e3f433,David,Williams,Robert Davis,Oncology,Chemotherapy,534.03,534.03,Cash,Failed
T008,A008,D010,P016,2023-05-24,08:45:00,Consultation,Cancelled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,57f93870c1c3feba455ba046b386c5b2a605bbf4e300d10a6a9237262f84a617,Michael,Taylor,Linda Wilson,Oncology,Physiotherapy,3413.64,3413.64,Cash,Failed
T009,A009,D010,P039,2023-03-05,13:45:00,Follow-up,Scheduled,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,21e65c5fe3199f0cdfddf55c8eeb6d4a39b261683c40ec08a367ecfbb9f489f6,Jane,Wilson,Linda Wilson,Oncology,Physiotherapy,4541.14,4541.14,Credit Card,Paid
T010,A010,D003,P005,2023-01-13,15:30:00,Therapy,Completed,true,true,true,true,BATCH_20260810_144925_920fea,SRC002,appointments,appointments.csv,2026-08-10T14:49:32.693Z,2026-08-10,f806fc8f83616deb6e5e5f6938c43a911d63205973babce3d539d370284b8521,David,Wilson,Jane Smith,Pediatrics,Physiotherapy,1595.67,1595.67,Cash,Paid


In [0]:




gold_appointment_summary = (
    silver_appointments
    .groupBy("appointment_id")
    .agg(
        F.first("patient_id").alias("patient_id"),
        F.first("doctor_id").alias("doctor_id"),
        F.first("appointment_date").alias("appointment_date"),
        F.first("appointment_time").alias("appointment_time"),
        F.first("reason_for_visit").alias("reason_for_visit"),
        F.first("status").alias("status")
    )
)

print(
    "Gold appointment summary rows:",
    gold_appointment_summary.count()
)

display(gold_appointment_summary)

Gold appointment summary rows: 200


appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status
A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled
A002,P032,D004,2023-06-09,14:30:00,Therapy,No-show
A003,P048,D004,2023-06-28,08:00:00,Consultation,Cancelled
A004,P025,D006,2023-09-01,09:15:00,Consultation,Cancelled
A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show
A006,P045,D006,2023-06-19,16:15:00,Checkup,Scheduled
A007,P001,D007,2023-04-09,10:30:00,Consultation,Scheduled
A008,P016,D010,2023-05-24,08:45:00,Consultation,Cancelled
A009,P039,D010,2023-03-05,13:45:00,Follow-up,Scheduled
A010,P005,D003,2023-01-13,15:30:00,Therapy,Completed


In [0]:
gold_appointment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.gold_appointment_summary"
    )

print("gold_appointment_summary created successfully")

gold_appointment_summary created successfully


In [0]:
gold_appointment_summary_check = spark.table(
    "healthcare.default.gold_appointment_summary"
)

print(
    "Gold appointment summary rows:",
    gold_appointment_summary_check.count()
)

display(gold_appointment_summary_check)

Gold appointment summary rows: 200


appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status
A012,P029,D003,2023-05-07,10:00:00,Follow-up,Completed
A044,P031,D006,2023-09-20,12:30:00,Follow-up,Completed
A047,P032,D007,2023-05-02,11:00:00,Therapy,Completed
A050,P045,D008,2023-08-16,15:00:00,Consultation,No-show
A078,P013,D008,2023-09-17,11:15:00,Consultation,No-show
A099,P011,D007,2023-07-04,15:00:00,Checkup,Completed
A124,P013,D008,2023-03-16,17:15:00,Emergency,Cancelled
A140,P012,D005,2023-02-05,15:15:00,Checkup,No-show
A143,P012,D007,2023-09-21,12:15:00,Checkup,Cancelled
A159,P016,D003,2023-04-08,16:15:00,Emergency,No-show


In [0]:
doctor_performance = (
    appointments
    .groupBy("doctor_id")
    .agg(
        F.count("appointment_id").alias("total_appointments"),

        F.sum(
            F.when(
                F.col("status") == "Completed",
                1
            ).otherwise(0)
        ).alias("completed_appointments"),

        F.sum(
            F.when(
                F.col("status") == "Cancelled",
                1
            ).otherwise(0)
        ).alias("cancelled_appointments"),

        F.sum(
            F.when(
                F.col("status") == "No-show",
                1
            ).otherwise(0)
        ).alias("no_show_appointments")
    )
    .join(
        doctors.select(
            "doctor_id",
            F.concat_ws(
                " ",
                F.col("first_name"),
                F.col("last_name")
            ).alias("doctor_name"),
            "specialization",
            "hospital_branch",
            "years_experience"
        ),
        on="doctor_id",
        how="left"
    )
)

display(
    doctor_performance
    .orderBy("doctor_id")
)

doctor_id,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,doctor_name,specialization,hospital_branch,years_experience
D001,25,6,7,7,David Taylor,Dermatology,Westside Clinic,17
D002,21,5,8,1,Jane Davis,Pediatrics,Eastside Clinic,24
D003,22,6,4,7,Jane Smith,Pediatrics,Eastside Clinic,19
D004,14,3,3,5,David Jones,Pediatrics,Central Hospital,28
D005,29,4,6,9,Sarah Taylor,Dermatology,Central Hospital,26
D006,24,5,6,6,Alex Davis,Pediatrics,Central Hospital,23
D007,13,5,5,2,Robert Davis,Oncology,Westside Clinic,26
D008,16,4,5,4,Linda Brown,Dermatology,Westside Clinic,5
D009,17,3,4,6,Sarah Smith,Pediatrics,Central Hospital,26
D010,19,5,3,5,Linda Wilson,Oncology,Eastside Clinic,21


In [0]:
doctor_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.gold_doctor_performance"
    )

print("gold_doctor_performance created successfully")

gold_doctor_performance created successfully


In [0]:
gold_doctor_performance = spark.table(
    "healthcare.default.gold_doctor_performance"
)

print(
    "Gold doctor performance rows:",
    gold_doctor_performance.count()
)

display(
    gold_doctor_performance
    .orderBy("doctor_id")
)

Gold doctor performance rows: 10


doctor_id,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,doctor_name,specialization,hospital_branch,years_experience
D001,25,6,7,7,David Taylor,Dermatology,Westside Clinic,17
D002,21,5,8,1,Jane Davis,Pediatrics,Eastside Clinic,24
D003,22,6,4,7,Jane Smith,Pediatrics,Eastside Clinic,19
D004,14,3,3,5,David Jones,Pediatrics,Central Hospital,28
D005,29,4,6,9,Sarah Taylor,Dermatology,Central Hospital,26
D006,24,5,6,6,Alex Davis,Pediatrics,Central Hospital,23
D007,13,5,5,2,Robert Davis,Oncology,Westside Clinic,26
D008,16,4,5,4,Linda Brown,Dermatology,Westside Clinic,5
D009,17,3,4,6,Sarah Smith,Pediatrics,Central Hospital,26
D010,19,5,3,5,Linda Wilson,Oncology,Eastside Clinic,21


In [0]:
patient_treatment_summary = (
    treatments
    .join(
        appointments.select(
            "appointment_id",
            "patient_id"
        ),
        on="appointment_id",
        how="left"
    )
    .groupBy("patient_id")
    .agg(
        F.count("treatment_id").alias("total_treatments"),
        F.sum("cost").alias("total_treatment_cost"),
        F.avg("cost").alias("average_treatment_cost"),
        F.min("treatment_date").alias("first_treatment_date"),
        F.max("treatment_date").alias("last_treatment_date")
    )
)

display(
    patient_treatment_summary
    .orderBy("patient_id")
)


patient_id,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,8,18609.91,2326.23875,2023-01-01,2023-11-14
P007,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28
P011,2,7728.880000000001,3864.4400000000005,2023-04-18,2023-07-04


In [0]:
patient_treatment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.gold_patient_treatment_summary"
    )

print("gold_patient_treatment_summary created successfully")

gold_patient_treatment_summary created successfully


In [0]:
gold_patient_treatment_summary = spark.table(
    "healthcare.default.gold_patient_treatment_summary"
)

print(
    "Gold patient treatment summary rows:",
    gold_patient_treatment_summary.count()
)

display(
    gold_patient_treatment_summary
    .orderBy("patient_id")
)

Gold patient treatment summary rows: 48


patient_id,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,8,18609.91,2326.23875,2023-01-01,2023-11-14
P007,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28
P011,2,7728.880000000001,3864.4400000000005,2023-04-18,2023-07-04


In [0]:
patient_billing_summary = (
    billing
    .groupBy("patient_id")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_billed_amount"),
        F.avg("amount").alias("average_bill_amount"),

        F.sum(
            F.when(
                F.col("payment_status") == "Paid",
                1
            ).otherwise(0)
        ).alias("paid_bills"),

        F.sum(
            F.when(
                F.col("payment_status") == "Pending",
                1
            ).otherwise(0)
        ).alias("pending_bills"),

        F.sum(
            F.when(
                F.col("payment_status") == "Failed",
                1
            ).otherwise(0)
        ).alias("failed_bills")
    )
)

display(
    patient_billing_summary
    .orderBy("patient_id")
)

patient_id,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills
P001,4,7719.069999999999,1929.7674999999997,1,1,2
P002,3,5968.74,1989.58,1,1,1
P003,2,7936.88,3968.44,1,1,0
P004,2,5362.51,2681.255,0,1,1
P005,8,18609.91,2326.23875,2,2,4
P007,4,10734.38,2683.595,3,0,1
P008,2,3636.8900000000003,1818.4450000000002,1,1,0
P009,4,10556.54,2639.135,1,2,1
P010,6,15929.149999999998,2654.858333333333,3,1,2
P011,2,7728.880000000001,3864.4400000000005,0,1,1


In [0]:
patient_billing_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.gold_patient_billing_summary"
    )

print("gold_patient_billing_summary created successfully")

gold_patient_billing_summary = spark.table(
    "healthcare.default.gold_patient_billing_summary"
)

print(
    "Gold patient billing summary rows:",
    gold_patient_billing_summary.count()
)

display(
    gold_patient_billing_summary
    .orderBy("patient_id")
)

gold_patient_billing_summary created successfully
Gold patient billing summary rows: 48


patient_id,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills
P001,4,7719.069999999999,1929.7674999999997,1,1,2
P002,3,5968.74,1989.58,1,1,1
P003,2,7936.88,3968.44,1,1,0
P004,2,5362.51,2681.255,0,1,1
P005,8,18609.91,2326.23875,2,2,4
P007,4,10734.38,2683.595,3,0,1
P008,2,3636.8900000000003,1818.4450000000002,1,1,0
P009,4,10556.54,2639.135,1,2,1
P010,6,15929.149999999998,2654.858333333333,3,1,2
P011,2,7728.880000000001,3864.4400000000005,0,1,1


In [0]:
patient_360_summary = (
    spark.table(
        "healthcare.default.gold_patient_summary"
    )
    .join(
        spark.table(
            "healthcare.default.gold_patient_treatment_summary"
        ),
        on="patient_id",
        how="left"
    )
    .join(
        spark.table(
            "healthcare.default.gold_patient_billing_summary"
        ),
        on="patient_id",
        how="left"
    )
)

display(
    patient_360_summary
    .orderBy("patient_id")
)

patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills
P001,David,Williams,Female,1955-06-04,2022-06-23,WellnessCorp,4,0,1,1,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09,4,7719.07,1929.7675,2023-01-16,2023-04-09,4,7719.07,1929.7675,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2
P002,Emily,Smith,Female,1984-10-12,2022-01-15,PulseSecure,3,0,1,0,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06,3,5968.74,1989.58,2023-01-20,2023-10-06,3,5968.74,1989.58,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1
P003,Laura,Jones,Male,1977-08-21,2022-02-07,PulseSecure,2,0,0,0,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26,2,7936.88,3968.44,2023-08-16,2023-08-26,2,7936.88,3968.44,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0
P004,Michael,Johnson,Female,1981-02-20,2021-03-02,HealthIndia,2,2,0,0,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07,2,5362.51,2681.255,2023-02-04,2023-07-07,2,5362.51,2681.255,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1
P005,David,Wilson,Male,1960-06-23,2021-09-29,MedCare Plus,8,2,0,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2023-01-01,2023-11-14,8,18609.91,2326.23875,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4
P006,Linda,Jones,Male,1963-06-16,2022-10-02,HealthIndia,0,0,0,0,null,null,0,0.0,0.0,0,0,0,null,null,0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null
P007,Alex,Johnson,Female,1989-06-08,2021-12-25,MedCare Plus,4,0,3,0,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1
P008,David,Davis,Female,1976-07-05,2021-05-25,WellnessCorp,2,1,0,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0
P009,Laura,Davis,Male,1971-12-11,2022-09-18,PulseSecure,4,1,1,1,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22,4,10556.54,2639.135,2023-03-21,2023-10-22,4,10556.54,2639.135,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1
P010,Michael,Taylor,Male,2001-10-13,2022-08-24,WellnessCorp,6,1,1,3,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2


In [0]:
from pyspark.sql import functions as F

gold_patient_base = spark.table(
    "healthcare.default.gold_patient_summary"
)

gold_patient_treatment = spark.table(
    "healthcare.default.gold_patient_treatment_summary"
)

gold_patient_billing = spark.table(
    "healthcare.default.gold_patient_billing_summary"
)

# Find columns that are not already present in the base patient summary
base_columns = set(gold_patient_base.columns)

treatment_columns = [
    c for c in gold_patient_treatment.columns
    if c == "patient_id" or c not in base_columns
]

billing_columns = [
    c for c in gold_patient_billing.columns
    if c == "patient_id" or c not in base_columns
]

patient_360_summary = (
    gold_patient_base
    .join(
        gold_patient_treatment.select(treatment_columns),
        on="patient_id",
        how="left"
    )
    .join(
        gold_patient_billing.select(billing_columns),
        on="patient_id",
        how="left"
    )
)

print(
    "Patient 360 summary rows:",
    patient_360_summary.count()
)

print(
    "Patient 360 summary columns:",
    len(patient_360_summary.columns)
)

display(
    patient_360_summary
    .orderBy("patient_id")
)

Patient 360 summary rows: 50
Patient 360 summary columns: 26


patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,David,Williams,Female,1955-06-04,2022-06-23,WellnessCorp,4,0,1,1,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,Emily,Smith,Female,1984-10-12,2022-01-15,PulseSecure,3,0,1,0,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,Laura,Jones,Male,1977-08-21,2022-02-07,PulseSecure,2,0,0,0,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,Michael,Johnson,Female,1981-02-20,2021-03-02,HealthIndia,2,2,0,0,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,David,Wilson,Male,1960-06-23,2021-09-29,MedCare Plus,8,2,0,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2023-01-01,2023-11-14
P006,Linda,Jones,Male,1963-06-16,2022-10-02,HealthIndia,0,0,0,0,null,null,0,0.0,0.0,0,0,0,null,null,0,0.0,0.0,null,null
P007,Alex,Johnson,Female,1989-06-08,2021-12-25,MedCare Plus,4,0,3,0,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,David,Davis,Female,1976-07-05,2021-05-25,WellnessCorp,2,1,0,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,Laura,Davis,Male,1971-12-11,2022-09-18,PulseSecure,4,1,1,1,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,Michael,Taylor,Male,2001-10-13,2022-08-24,WellnessCorp,6,1,1,3,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28


In [0]:
patient_360_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare.default.gold_patient_360_summary"
    )

print("gold_patient_360_summary created successfully")

gold_patient_360_summary created successfully


In [0]:
gold_patient_360_check = spark.table(
    "healthcare.default.gold_patient_360_summary"
)

print(
    "Final Gold patient 360 rows:",
    gold_patient_360_check.count()
)

display(
    gold_patient_360_check
    .orderBy("patient_id")
)

Final Gold patient 360 rows: 50


patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,David,Williams,Female,1955-06-04,2022-06-23,WellnessCorp,4,0,1,1,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,Emily,Smith,Female,1984-10-12,2022-01-15,PulseSecure,3,0,1,0,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,Laura,Jones,Male,1977-08-21,2022-02-07,PulseSecure,2,0,0,0,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,Michael,Johnson,Female,1981-02-20,2021-03-02,HealthIndia,2,2,0,0,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,David,Wilson,Male,1960-06-23,2021-09-29,MedCare Plus,8,2,0,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2023-01-01,2023-11-14
P006,Linda,Jones,Male,1963-06-16,2022-10-02,HealthIndia,0,0,0,0,null,null,0,0.0,0.0,0,0,0,null,null,0,0.0,0.0,null,null
P007,Alex,Johnson,Female,1989-06-08,2021-12-25,MedCare Plus,4,0,3,0,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,David,Davis,Female,1976-07-05,2021-05-25,WellnessCorp,2,1,0,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,Laura,Davis,Male,1971-12-11,2022-09-18,PulseSecure,4,1,1,1,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,Michael,Taylor,Male,2001-10-13,2022-08-24,WellnessCorp,6,1,1,3,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28


In [0]:

gold_patient_360 = spark.table(
    "healthcare.default.gold_patient_360_summary"
)

print(
    "Final Gold Patient 360 rows:",
    gold_patient_360.count()
)

print(
    "Final Gold Patient 360 columns:",
    len(gold_patient_360.columns)
)

Final Gold Patient 360 rows: 50
Final Gold Patient 360 columns: 26


In [0]:
duplicate_patients = (
    gold_patient_360
    .groupBy("patient_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate patient IDs:",
    duplicate_patients.count()
)

display(duplicate_patients)

Duplicate patient IDs: 0


patient_id,count


In [0]:
null_patient_ids = gold_patient_360.filter(
    F.col("patient_id").isNull()
)

print(
    "Null patient IDs:",
    null_patient_ids.count()
)

display(null_patient_ids)

Null patient IDs: 0


patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date


In [0]:
display(
    gold_patient_360
    .orderBy("patient_id")
)

patient_id,first_name,last_name,gender,date_of_birth,registration_date,insurance_provider,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,first_appointment_date,last_appointment_date,total_bills,total_billed_amount,average_bill_amount,paid_bills,pending_bills,failed_bills,first_bill_date,last_bill_date,total_treatments,total_treatment_cost,average_treatment_cost,first_treatment_date,last_treatment_date
P001,David,Williams,Female,1955-06-04,2022-06-23,WellnessCorp,4,0,1,1,2023-01-16,2023-04-09,4,7719.069999999999,1929.7674999999997,1,1,2,2023-01-16,2023-04-09,4,7719.07,1929.7675,2023-01-16,2023-04-09
P002,Emily,Smith,Female,1984-10-12,2022-01-15,PulseSecure,3,0,1,0,2023-01-20,2023-10-06,3,5968.74,1989.58,1,1,1,2023-01-20,2023-10-06,3,5968.74,1989.58,2023-01-20,2023-10-06
P003,Laura,Jones,Male,1977-08-21,2022-02-07,PulseSecure,2,0,0,0,2023-08-16,2023-08-26,2,7936.88,3968.44,1,1,0,2023-08-16,2023-08-26,2,7936.88,3968.44,2023-08-16,2023-08-26
P004,Michael,Johnson,Female,1981-02-20,2021-03-02,HealthIndia,2,2,0,0,2023-02-04,2023-07-07,2,5362.51,2681.255,0,1,1,2023-02-04,2023-07-07,2,5362.51,2681.255,2023-02-04,2023-07-07
P005,David,Wilson,Male,1960-06-23,2021-09-29,MedCare Plus,8,2,0,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2,2,4,2023-01-01,2023-11-14,8,18609.91,2326.23875,2023-01-01,2023-11-14
P006,Linda,Jones,Male,1963-06-16,2022-10-02,HealthIndia,0,0,0,0,null,null,0,0.0,0.0,0,0,0,null,null,0,0.0,0.0,null,null
P007,Alex,Johnson,Female,1989-06-08,2021-12-25,MedCare Plus,4,0,3,0,2023-01-07,2023-12-30,4,10734.38,2683.595,3,0,1,2023-01-07,2023-12-30,4,10734.380000000001,2683.5950000000003,2023-01-07,2023-12-30
P008,David,Davis,Female,1976-07-05,2021-05-25,WellnessCorp,2,1,0,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,1,1,0,2023-04-06,2023-05-02,2,3636.8900000000003,1818.4450000000002,2023-04-06,2023-05-02
P009,Laura,Davis,Male,1971-12-11,2022-09-18,PulseSecure,4,1,1,1,2023-03-21,2023-10-22,4,10556.54,2639.135,1,2,1,2023-03-21,2023-10-22,4,10556.54,2639.135,2023-03-21,2023-10-22
P010,Michael,Taylor,Male,2001-10-13,2022-08-24,WellnessCorp,6,1,1,3,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,3,1,2,2023-03-27,2023-09-28,6,15929.149999999998,2654.858333333333,2023-03-27,2023-09-28


In [0]:
revenue_summary = (
    billing
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount"),
        F.min("amount").alias("minimum_bill_amount"),
        F.max("amount").alias("maximum_bill_amount")
    )
)

display(revenue_summary)

total_bills,total_revenue,average_bill_amount,minimum_bill_amount,maximum_bill_amount
200,551249.85,2756.24925,534.03,4973.63


In [0]:
revenue_by_status = (
    billing
    .groupBy("payment_status")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("payment_status")
)

display(revenue_by_status)

payment_status,total_bills,total_revenue,average_bill_amount
Failed,67,193212.94000000006,2883.775223880598
Paid,64,173424.9,2709.7640625
Pending,69,184612.00999999998,2675.536376811594


In [0]:
revenue_by_method = (
    billing
    .groupBy("payment_method")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("payment_method")
)

display(revenue_by_method)

payment_method,total_bills,total_revenue,average_bill_amount
Cash,61,167707.14,2749.2973770491803
Credit Card,75,201382.43000000002,2685.099066666667
Insurance,64,182160.28,2846.254375


In [0]:
revenue_by_month = (
    billing
    .withColumn(
        "month",
        F.month("bill_date")
    )
    .groupBy("month")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("month")
)

display(revenue_by_month)


month,total_bills,total_revenue,average_bill_amount
1,20,58701.23000000001,2935.0615000000007
2,14,36669.689999999995,2619.263571428571
3,19,47304.29,2489.6994736842107
4,25,64271.53999999998,2570.8615999999993
5,19,48791.05,2567.9500000000003
6,18,56887.82,3160.4344444444446
7,16,39880.19,2492.511875
8,15,41958.67,2797.2446666666665
9,11,33426.53,3038.7754545454545
10,14,43314.15,3093.8678571428572


In [0]:
revenue_by_insurance = (
    billing
    .join(
        patients.select(
            "patient_id",
            "insurance_provider"
        ),
        on="patient_id",
        how="left"
    )
    .groupBy("insurance_provider")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("insurance_provider")
)

display(revenue_by_insurance)

insurance_provider,total_bills,total_revenue,average_bill_amount
HealthIndia,22,53823.759999999995,2446.5345454545454
MedCare Plus,84,241092.29,2870.1463095238096
PulseSecure,36,104473.22000000002,2902.033888888889
WellnessCorp,58,151860.58000000002,2618.285862068966


In [0]:
revenue_by_branch = (
    billing
    .join(
        treatments.select(
            "treatment_id",
            "appointment_id"
        ),
        on="treatment_id",
        how="left"
    )
    .join(
        appointments.select(
            "appointment_id",
            "doctor_id"
        ),
        on="appointment_id",
        how="left"
    )
    .join(
        doctors.select(
            "doctor_id",
            "hospital_branch"
        ),
        on="doctor_id",
        how="left"
    )
    .groupBy("hospital_branch")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("hospital_branch")
)

display(revenue_by_branch)

hospital_branch,total_bills,total_revenue,average_bill_amount
Central Hospital,84,229039.43999999997,2726.66
Eastside Clinic,62,162031.09999999998,2613.404838709677
Westside Clinic,54,160179.31,2966.2835185185186


In [0]:
revenue_by_treatment = (
    billing
    .join(
        treatments.select(
            "treatment_id",
            "treatment_type"
        ),
        on="treatment_id",
        how="left"
    )
    .groupBy("treatment_type")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("treatment_type")
)

display(revenue_by_treatment)

treatment_type,total_bills,total_revenue,average_bill_amount
Chemotherapy,49,128855.68000000002,2629.707755102041
Ecg,38,96224.24,2532.2168421052634
Mri,36,116098.16,3224.948888888889
Physiotherapy,36,99418.09999999999,2761.6138888888886
X-ray,41,110653.66999999998,2698.8699999999994


In [0]:
revenue_by_doctor = (
    billing
    .join(
        treatments.select(
            "treatment_id",
            "appointment_id"
        ),
        on="treatment_id",
        how="left"
    )
    .join(
        appointments.select(
            "appointment_id",
            "doctor_id"
        ),
        on="appointment_id",
        how="left"
    )
    .join(
        doctors.select(
            "doctor_id",
            "first_name",
            "last_name",
            "specialization"
        ),
        on="doctor_id",
        how="left"
    )
    .groupBy(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization"
    )
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("doctor_id")
)

display(revenue_by_doctor)

doctor_id,first_name,last_name,specialization,total_bills,total_revenue,average_bill_amount
D001,David,Taylor,Dermatology,25,66585.39,2663.4156
D002,Jane,Davis,Pediatrics,21,59803.45999999999,2847.783809523809
D003,Jane,Smith,Pediatrics,22,52791.409999999996,2399.6095454545452
D004,David,Jones,Pediatrics,14,39315.950000000004,2808.2821428571433
D005,Sarah,Taylor,Dermatology,29,82696.48,2851.6027586206897
D006,Alex,Davis,Pediatrics,24,69586.09999999999,2899.420833333333
D007,Robert,Davis,Oncology,13,40166.5,3089.730769230769
D008,Linda,Brown,Dermatology,16,53427.42,3339.21375
D009,Sarah,Smith,Pediatrics,17,37440.91,2202.4064705882356
D010,Linda,Wilson,Oncology,19,49436.229999999996,2601.906842105263


In [0]:
revenue_by_method = (
    billing
    .groupBy("payment_method")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("payment_method")
)

display(revenue_by_method)

payment_method,total_bills,total_revenue,average_bill_amount
Cash,61,167707.14,2749.2973770491803
Credit Card,75,201382.43000000002,2685.099066666667
Insurance,64,182160.28,2846.254375


In [0]:
revenue_by_month = (
    billing
    .withColumn("month", F.month("bill_date"))
    .groupBy("month")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("month")
)

display(revenue_by_month)

month,total_bills,total_revenue,average_bill_amount
1,20,58701.23000000001,2935.0615000000007
2,14,36669.689999999995,2619.263571428571
3,19,47304.29,2489.6994736842107
4,25,64271.53999999998,2570.8615999999993
5,19,48791.05,2567.9500000000003
6,18,56887.82,3160.4344444444446
7,16,39880.19,2492.511875
8,15,41958.67,2797.2446666666665
9,11,33426.53,3038.7754545454545
10,14,43314.15,3093.8678571428572


In [0]:
revenue_by_year = (
    billing
    .withColumn("year", F.year("bill_date"))
    .groupBy("year")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("year")
)

display(revenue_by_year)

year,total_bills,total_revenue,average_bill_amount
2023,200,551249.85,2756.24925


In [0]:
revenue_by_status = (
    billing
    .groupBy("payment_status")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("payment_status")
)

display(revenue_by_status)

payment_status,total_bills,total_revenue,average_bill_amount
Failed,67,193212.94000000006,2883.775223880598
Paid,64,173424.9,2709.7640625
Pending,69,184612.00999999998,2675.536376811594


In [0]:
top_patients_by_billing = (
    billing
    .groupBy("patient_id")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_billed_amount"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy(F.desc("total_billed_amount"))
    .limit(10)
)

display(top_patients_by_billing)

patient_id,total_bills,total_billed_amount,average_bill_amount
P012,10,30053.08,3005.308
P049,7,23554.060000000005,3364.865714285715
P016,7,22967.940000000002,3281.134285714286
P036,7,21583.56,3083.3657142857146
P025,5,19513.170000000002,3902.6340000000005
P005,8,18609.91,2326.23875
P035,7,18407.420000000002,2629.631428571429
P048,5,17082.479999999996,3416.495999999999
P010,6,15929.149999999998,2654.858333333333
P017,4,14850.28,3712.57


In [0]:
top_patients_by_count = (
    billing
    .groupBy("patient_id")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_billed_amount"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy(
        F.desc("total_bills"),
        F.desc("total_billed_amount")
    )
    .limit(10)
)

display(top_patients_by_count)

patient_id,total_bills,total_billed_amount,average_bill_amount
P012,10,30053.08,3005.308
P005,8,18609.91,2326.23875
P049,7,23554.060000000005,3364.865714285715
P016,7,22967.940000000002,3281.134285714286
P036,7,21583.56,3083.3657142857146
P035,7,18407.420000000002,2629.631428571429
P029,7,13324.5,1903.5
P010,6,15929.149999999998,2654.858333333333
P037,6,13886.890000000001,2314.481666666667
P023,6,13237.689999999999,2206.2816666666663


In [0]:
top_patients_by_average = (
    billing
    .groupBy("patient_id")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_billed_amount"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy(F.desc("average_bill_amount"))
    .limit(10)
)

display(top_patients_by_average)

patient_id,total_bills,total_billed_amount,average_bill_amount
P044,2,9324.099999999999,4662.049999999999
P014,3,13236.190000000002,4412.063333333334
P003,2,7936.88,3968.44
P025,5,19513.170000000002,3902.6340000000005
P011,2,7728.880000000001,3864.4400000000005
P038,2,7537.98,3768.99
P017,4,14850.28,3712.57
P047,3,10423.17,3474.39
P031,4,13732.02,3433.005
P048,5,17082.479999999996,3416.495999999999


In [0]:
top_doctors_by_revenue = (
    billing
    .join(
        treatments.select(
            "treatment_id",
            "appointment_id"
        ),
        on="treatment_id",
        how="left"
    )
    .join(
        appointments.select(
            "appointment_id",
            "doctor_id"
        ),
        on="appointment_id",
        how="left"
    )
    .join(
        doctors.select(
            "doctor_id",
            "first_name",
            "last_name",
            "specialization"
        ),
        on="doctor_id",
        how="left"
    )
    .groupBy(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization"
    )
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)

display(top_doctors_by_revenue)

doctor_id,first_name,last_name,specialization,total_bills,total_revenue,average_bill_amount
D005,Sarah,Taylor,Dermatology,29,82696.48,2851.6027586206897
D006,Alex,Davis,Pediatrics,24,69586.09999999999,2899.420833333333
D001,David,Taylor,Dermatology,25,66585.39,2663.4156
D002,Jane,Davis,Pediatrics,21,59803.45999999999,2847.783809523809
D008,Linda,Brown,Dermatology,16,53427.42,3339.21375
D003,Jane,Smith,Pediatrics,22,52791.409999999996,2399.6095454545452
D010,Linda,Wilson,Oncology,19,49436.229999999996,2601.906842105263
D007,Robert,Davis,Oncology,13,40166.5,3089.730769230769
D004,David,Jones,Pediatrics,14,39315.950000000004,2808.2821428571433
D009,Sarah,Smith,Pediatrics,17,37440.91,2202.4064705882356


In [0]:
top_doctors_by_average = (
    billing
    .join(
        treatments.select(
            "treatment_id",
            "appointment_id"
        ),
        on="treatment_id",
        how="left"
    )
    .join(
        appointments.select(
            "appointment_id",
            "doctor_id"
        ),
        on="appointment_id",
        how="left"
    )
    .join(
        doctors.select(
            "doctor_id",
            "first_name",
            "last_name",
            "specialization"
        ),
        on="doctor_id",
        how="left"
    )
    .groupBy(
        "doctor_id",
        "first_name",
        "last_name",
        "specialization"
    )
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy(F.desc("average_bill_amount"))
    .limit(10)
)

display(top_doctors_by_average)

doctor_id,first_name,last_name,specialization,total_bills,total_revenue,average_bill_amount
D008,Linda,Brown,Dermatology,16,53427.42,3339.21375
D007,Robert,Davis,Oncology,13,40166.5,3089.730769230769
D006,Alex,Davis,Pediatrics,24,69586.09999999999,2899.420833333333
D005,Sarah,Taylor,Dermatology,29,82696.48,2851.6027586206897
D002,Jane,Davis,Pediatrics,21,59803.45999999999,2847.783809523809
D004,David,Jones,Pediatrics,14,39315.950000000004,2808.2821428571433
D001,David,Taylor,Dermatology,25,66585.39,2663.4156
D010,Linda,Wilson,Oncology,19,49436.229999999996,2601.906842105263
D003,Jane,Smith,Pediatrics,22,52791.409999999996,2399.6095454545452
D009,Sarah,Smith,Pediatrics,17,37440.91,2202.4064705882356


In [0]:
revenue_by_specialization = (
    billing
    .join(
        treatments.select(
            "treatment_id",
            "appointment_id"
        ),
        on="treatment_id",
        how="left"
    )
    .join(
        appointments.select(
            "appointment_id",
            "doctor_id"
        ),
        on="appointment_id",
        how="left"
    )
    .join(
        doctors.select(
            "doctor_id",
            "specialization"
        ),
        on="doctor_id",
        how="left"
    )
    .groupBy("specialization")
    .agg(
        F.count("bill_id").alias("total_bills"),
        F.sum("amount").alias("total_revenue"),
        F.avg("amount").alias("average_bill_amount")
    )
    .orderBy("specialization")
)

display(revenue_by_specialization)

specialization,total_bills,total_revenue,average_bill_amount
Dermatology,70,202709.28999999998,2895.8469999999998
Oncology,32,89602.73,2800.0853125
Pediatrics,98,258937.83,2642.2227551020405


In [0]:
revenue_by_insurance_ranked = (
    revenue_by_insurance
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_insurance_ranked)

insurance_provider,total_bills,total_revenue,average_bill_amount
MedCare Plus,84,241092.29,2870.1463095238096
WellnessCorp,58,151860.58000000002,2618.285862068966
PulseSecure,36,104473.22000000002,2902.033888888889
HealthIndia,22,53823.759999999995,2446.5345454545454


In [0]:
revenue_by_branch_ranked = (
    revenue_by_branch
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_branch_ranked)

hospital_branch,total_bills,total_revenue,average_bill_amount
Central Hospital,84,229039.43999999997,2726.66
Eastside Clinic,62,162031.09999999998,2613.404838709677
Westside Clinic,54,160179.31,2966.2835185185186


In [0]:
revenue_by_treatment_ranked = (
    revenue_by_treatment
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_treatment_ranked)

treatment_type,total_bills,total_revenue,average_bill_amount
Chemotherapy,49,128855.68000000002,2629.707755102041
Mri,36,116098.16,3224.948888888889
X-ray,41,110653.66999999998,2698.8699999999994
Physiotherapy,36,99418.09999999999,2761.6138888888886
Ecg,38,96224.24,2532.2168421052634


In [0]:
revenue_by_status_ranked = (
    revenue_by_status
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_status_ranked)

payment_status,total_bills,total_revenue,average_bill_amount
Failed,67,193212.94000000006,2883.775223880598
Pending,69,184612.00999999998,2675.536376811594
Paid,64,173424.9,2709.7640625


In [0]:
revenue_by_month_ranked = (
    revenue_by_month
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_month_ranked)

month,total_bills,total_revenue,average_bill_amount
4,25,64271.53999999998,2570.8615999999993
1,20,58701.23000000001,2935.0615000000007
6,18,56887.82,3160.4344444444446
11,17,52474.98000000001,3086.7635294117654
5,19,48791.05,2567.9500000000003
3,19,47304.29,2489.6994736842107
10,14,43314.15,3093.8678571428572
8,15,41958.67,2797.2446666666665
7,16,39880.19,2492.511875
2,14,36669.689999999995,2619.263571428571


In [0]:
revenue_by_year_ranked = (
    revenue_by_year
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_year_ranked)

year,total_bills,total_revenue,average_bill_amount
2023,200,551249.85,2756.24925


In [0]:
revenue_by_year_ranked = (
    revenue_by_year
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_year_ranked)

year,total_bills,total_revenue,average_bill_amount
2023,200,551249.85,2756.24925


In [0]:
top_revenue_month = (
    revenue_by_month
    .orderBy(F.desc("total_revenue"))
    .limit(1)
)

display(top_revenue_month)

month,total_bills,total_revenue,average_bill_amount
4,25,64271.53999999998,2570.8615999999993


In [0]:
top_revenue_doctor = (
    revenue_by_doctor
    .orderBy(F.desc("total_revenue"))
    .limit(1)
)

display(top_revenue_doctor)

doctor_id,first_name,last_name,specialization,total_bills,total_revenue,average_bill_amount
D005,Sarah,Taylor,Dermatology,29,82696.48,2851.6027586206897


In [0]:
top_average_doctor = (
    revenue_by_doctor
    .orderBy(F.desc("average_bill_amount"))
    .limit(1)
)

display(top_average_doctor)

doctor_id,first_name,last_name,specialization,total_bills,total_revenue,average_bill_amount
D008,Linda,Brown,Dermatology,16,53427.42,3339.21375


In [0]:
top_revenue_treatment = (
    revenue_by_treatment
    .orderBy(F.desc("total_revenue"))
    .limit(1)
)

display(top_revenue_treatment)

treatment_type,total_bills,total_revenue,average_bill_amount
Chemotherapy,49,128855.68000000002,2629.707755102041


In [0]:
top_average_treatment = (
    revenue_by_treatment
    .orderBy(F.desc("average_bill_amount"))
    .limit(1)
)

display(top_average_treatment)

treatment_type,total_bills,total_revenue,average_bill_amount
Mri,36,116098.16,3224.948888888889


In [0]:
top_revenue_insurance = (
    revenue_by_insurance
    .orderBy(F.desc("total_revenue"))
    .limit(1)
)

display(top_revenue_insurance)

insurance_provider,total_bills,total_revenue,average_bill_amount
MedCare Plus,84,241092.29,2870.1463095238096


In [0]:
top_average_insurance = (
    revenue_by_insurance
    .orderBy(F.desc("average_bill_amount"))
    .limit(1)
)

display(top_average_insurance)

insurance_provider,total_bills,total_revenue,average_bill_amount
PulseSecure,36,104473.22000000002,2902.033888888889
